# Embedding-Based Job Search - Exercise

**Business problem:** rank advertisements that match a user's query.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Compare BM25, retrieval-trained embeddings and rank fusion.
- Compute relevance metrics with explicit draft judgments.
- Connect retrieval quality to requirements and user usefulness.

## Dataset and practical scope

24 approved authentic German excerpts. Titles are anonymized and are not indexed. Four development and eight test queries have assistant-drafted judgments needing review. T03 was revised after inspecting earlier results: this is a revised teaching comparison, not untouched holdout evidence or recall over all 5,011 full ads.

Complete the `None` values marked `# TODO` before running the affected cells. Replace the answer prompts with one or two sentences.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Minimal data inspection

Only the quoted fragments are searched. The longer criterion describes our draft intent, not additional model input.

In [ ]:
import json
import re
from collections import Counter

panel = json.loads((DATA_ROOT / 'nlp_search_benchmark.json').read_text(encoding='utf-8'))
documents = pd.DataFrame([{'id': item['id'], 'title': item['title'],
                           'text': ' [...] '.join(item['fragments'])}
                          for item in panel['documents']])
queries = panel['queries']
display(documents.head(3))
print('Ads:', len(documents), '| Queries:', len(queries), '| Draft judgments:', panel['judgment_status'])

## 2. Lexical baseline: BM25

The supplied implementation reuses the lecture's term-count, rarity and length formula. It preserves identifiers such as C# and .NET. Zero-score ads are not used to fill an empty list; no exact matches is a valid result.

In [ ]:
def tokens(text):
    return re.findall(r"c\+\+|c#|\.net|[^\W_]+(?:[-'][^\W_]+)*", text.casefold())

counts = [Counter(tokens(text)) for text in documents['text']]
lengths = np.array([sum(row.values()) for row in counts], dtype=float)
df = Counter(term for row in counts for term in row)

def bm25_scores(query):
    scores = np.zeros(len(documents))
    for term in set(tokens(query)):
        frequency = np.array([row[term] for row in counts], dtype=float)
        idf = np.log1p((len(documents) - df[term] + 0.5) / (df[term] + 0.5))
        scores += idf * frequency * 2.2 / (frequency + 1.2 * (0.25 + 0.75 * lengths / lengths.mean()))
    return scores

def rank_ids(scores, k=3, positive_only=False):
    order = sorted(range(len(documents)), key=lambda i: (-scores[i], documents.iloc[i]['id']))
    return [documents.iloc[i]['id'] for i in order if not positive_only or scores[i] > 0][:k]

K, DEPTH = 3, 10
keyword_lists = {query['id']: rank_ids(bm25_scores(query['text']), DEPTH, True) for query in queries}
print('Exact keyword matches for data engineer:', keyword_lists['T03'])

## 3. Encode passages and queries

First use downloads public E5 files; text stays local. Prefixes are model-specific. Normalized vectors make dot product equal cosine. **Coding task:** compute all query-ad scores with `query_vectors @ ad_vectors.T`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(2)
MODEL = 'intfloat/multilingual-e5-small'
REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
search_encoder = SentenceTransformer(MODEL, revision=REVISION, device='cpu')
ad_vectors = search_encoder.encode(['passage: ' + text for text in documents['text']],
                                   normalize_embeddings=True, show_progress_bar=False)
query_vectors = search_encoder.encode(['query: ' + query['text'] for query in queries],
                                      normalize_embeddings=True, show_progress_bar=False)
print('Ad-vector shape:', ad_vectors.shape)
scores = None  # TODO: query-by-ad cosine score matrix
dense_lists = {query['id']: rank_ids(scores[i], DEPTH) for i, query in enumerate(queries)}

## 4. Combine candidate lists

**Coding task:** use `1 / (60 + position)` for each list's contribution. Missing results contribute nothing; a result missing from both lists cannot be recovered by fusion.

In [ ]:
def fuse(rankings, k=3):
    votes = {}
    for ranking in rankings:
        for position, ad in enumerate(ranking, 1):
            contribution = None  # TODO: rank contribution
            votes[ad] = votes.get(ad, 0) + contribution
    return sorted(votes, key=lambda ad: (-votes[ad], ad))[:k]

hybrid_lists = {query['id']: fuse([keyword_lists[query['id']], dense_lists[query['id']]])
                for query in queries}

## 5. Development evaluation

Precision divides by K even if a list has fewer hits. Recall divides by the number of draft-relevant excerpts. Reciprocal rank uses the first relevant result; averaging it gives MRR. Review the supplied code and judgments.

In [ ]:
def metrics(ranking, relevant, k=3):
    positions = [i for i, ad in enumerate(ranking[:k], 1) if ad in relevant]
    return {'P@3': len(positions) / k,
            'R@3': len(positions) / len(relevant),
            'RR@3': 1 / positions[0] if positions else 0.}

rows = []
recipes = {'BM25': keyword_lists, 'E5': dense_lists, 'RRF': hybrid_lists}
for query in queries:
    for name, lists in recipes.items():
        rows.append({'query': query['id'], 'split': query['split'], 'recipe': name,
                     **metrics(lists[query['id']], set(query['relevant']))})
measurements = pd.DataFrame(rows)
display(measurements[measurements['split'] == 'development']
        .groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean().rename(columns={'RR@3': 'MRR@3'}).round(3))

### Interpret the result

Why can a high score or agreement between methods still return a poor match?

**Your interpretation:** Replace this sentence with your answer.

## 6. Fixed-recipe test comparison

Keep K=3, candidate depth 10 and RRF constant 60 fixed. Do not retune after this table. These eight queries are not a pristine holdout.

In [ ]:
test = measurements[measurements['split'] == 'test']
display(test.groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean()
        .rename(columns={'RR@3': 'MRR@3'}).round(3))

## Interpretation: read evidence, not just the mean

Inspect the same query under every recipe. MATCH means our draft criterion is supported in the excerpt, not verified suitability.

In [ ]:
selected = next(query for query in queries if query['id'] == 'T03')
print('Query:', selected['text'])
print('Draft criterion:', selected['criterion'])
for name, rankings in recipes.items():
    ids = rankings[selected['id']][:K]
    hits = documents.set_index('id').loc[ids].copy()
    hits['draft match'] = [ad in selected['relevant'] for ad in ids]
    print(name)
    display(hits[['title', 'text', 'draft match']])

### Interpret the result

Which recipe deserves further testing? What would a deployment evaluation add?

**Your interpretation:** Replace this sentence with your answer.

## Summary and business implications

Offline relevance is not online user benefit. Missing excerpt evidence is not proof that the whole ad is irrelevant. Do not silently revise labels to make a model win.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric.

## Further exercises

- Inspect C# .NET (T07) and the shift query (T08).
- Can recall change while the first relevant rank stays fixed?
- Optional: compare plain pooled BERT, chunking or reranking on fresh reviewed data.

## References

**Required reading:** https://www.sbert.net/examples/applications/semantic-search/README.html
and https://huggingface.co/intfloat/multilingual-e5-small
**Instructor references:** Reimers & Gurevych (2019), *Sentence-BERT*, https://aclanthology.org/D19-1410/;
Wang et al. (2024), *Multilingual E5 Text Embeddings*, https://arxiv.org/abs/2402.05672;
Cormack et al. (2009), *Reciprocal rank fusion*, https://doi.org/10.1145/1571941.1572114.
**Official BM25 documentation:** https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.